## 1. Imports for this section



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay, classification_report,
                              roc_curve, roc_auc_score)

sns.set_theme(style="whitegrid")


## 2. Model Setup

**Concept — the three algorithms and why these hyperparameters:**

- **Logistic Regression** (`max_iter=1000`): a linear model — it learns one weight per feature and
  combines them into a probability via the sigmoid function.
- **Decision Tree** (`max_depth=5`): learns a sequence of yes/no splits on feature values.
- **Random Forest** (`n_estimators=200, max_depth=6`): trains 200 individual decision trees, each on
  a random subset of rows and features, and averages their votes.

`random_state=RANDOM_STATE` is set on every model so re-running this notebook produces identical
results every time — required for anyone (including the grader) to reproduce our numbers.


In [2]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE),
}


NameError: name 'RANDOM_STATE' is not defined

## 3. Cross-Validation Check, Then Final Training




In [3]:
cv_summary = []
trained_models = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring="accuracy")
    cv_summary.append({
        "Model": name,
        "CV Accuracy (mean)": cv_scores.mean(),
        "CV Accuracy (std)": cv_scores.std(),
    })
    model.fit(X_train_s, y_train)
    trained_models[name] = model

cv_df = pd.DataFrame(cv_summary).set_index("Model").round(3)
cv_df


NameError: name 'models' is not defined

## 4. Evaluation Metrics and Results

**Concept — why four metrics, not just accuracy:

- **Accuracy** — overall fraction correct (can be misleading alone, as above).
- **Precision** — of students predicted to pass, what fraction actually passed.
- **Recall** — of students who actually passed, what fraction did we correctly catch? (For this
  project, recall on the **fail** class specifically is the more operationally important number,
  since the whole point is catching at-risk students — see the classification reports below.)
- **F1-score** — harmonic mean of precision and recall, a single balanced number.


In [ ]:
results = []
for name, model in trained_models.items():
    pred = model.predict(X_test_s)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-score": f1_score(y_test, pred),
    })

results_df = pd.DataFrame(results).set_index("Model").round(3)
results_df


## 5. Confusion Matrices

**Concept — confusion matrix:** every metric above is computed from four counts — true negatives,
false positives, false negatives, true positives. Seeing the raw 2x2 table matters because two
models can have identical accuracy while making very different *kinds* of mistakes


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, model) in zip(axes, trained_models.items()):
    pred = model.predict(X_test_s)
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Fail", "Pass"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout()
plt.show()
